## About Guided Cursor

Welcome to **Guided Cursor**, an AI-powered platform designed to guide you through programming problems step by step. As with all AI systems, responses may not always be perfectly accurate. You are encouraged to think critically and verify all output independently.

**Privacy.** We may collect anonymised usage data to improve the platform and support academic research into AI-assisted pedagogy. No data will be shared outside the project team. Your usage and performance will **not** be disclosed to module leaders and will have **no bearing** on your academic grades.

**Data Retention.** This platform may be taken offline at the end of the academic term, and all stored data may be permanently deleted. Please back up any materials you wish to keep in advance.

**Contact.** For any questions or concerns, please reach out to **hello@guidedcursor.studio**.

# Core Concepts and Applications

**Learning objectives**

By the end of this notebook you will be able to:

- Compute determinants, inverses, and ranks of matrices
- Solve systems of linear equations with NumPy
- Find eigenvalues and eigenvectors, and verify eigendecomposition
- Apply matrix operations to real-world problems: encryption (Hill cipher) and web page ranking (PageRank)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
import networkx as nx
from linalg_hints import show_hint
from linalg_verify import (
    check_hill_encrypt, check_valid_key, check_hill_decrypt,
    check_pagerank, check_power_iteration,
)

## 1. Determinants, Inverses, and Rank

### 1.1 Determinant

The **determinant** of a $2 \times 2$ matrix is:

$$\det\begin{pmatrix} a & b \\ c & d \end{pmatrix} = ad - bc$$

A determinant of zero means the matrix is **singular** — it squashes space into a lower dimension and cannot be inverted.

In [ ]:
# 2x2 determinant: ad - bc = 1*4 - 2*3 = -2
A = np.array([[1, 2], [3, 4]])
print("det(A):", np.linalg.det(A))

# 3x3 — hard by hand, but one line in NumPy
B = np.array([[2, 1, 0], [1, 3, 1], [0, 2, 1]])
print("det(B):", np.linalg.det(B))

# Singular: second row is twice the first -> det = 0
S = np.array([[1, 2], [2, 4]])
print("det(S):", np.linalg.det(S))

In [ ]:
# Complete code — just run this cell
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Left: invertible matrix — parallelogram has area
A = np.array([[2, 1], [0, 1.5]])
col1, col2 = A[:, 0], A[:, 1]
verts = np.array([[0, 0], col1, col1 + col2, col2, [0, 0]])
axes[0].fill(verts[:, 0], verts[:, 1], alpha=0.3, color="steelblue")
axes[0].quiver(0, 0, *col1, angles="xy", scale_units="xy", scale=1, color="steelblue")
axes[0].quiver(0, 0, *col2, angles="xy", scale_units="xy", scale=1, color="coral")
axes[0].set_title(f"det = {np.linalg.det(A):.1f} (area = {abs(np.linalg.det(A)):.1f})")
axes[0].set_xlim(-0.5, 4)
axes[0].set_ylim(-0.5, 3)
axes[0].set_aspect("equal")
axes[0].grid(True, alpha=0.3)

# Right: singular matrix — columns are collinear, area = 0
S = np.array([[1, 2], [1, 2]])
col1, col2 = S[:, 0], S[:, 1]
axes[1].quiver(0, 0, *col1, angles="xy", scale_units="xy", scale=1, color="steelblue", label="col 1")
axes[1].quiver(0, 0, *col2, angles="xy", scale_units="xy", scale=1, color="coral", label="col 2")
axes[1].set_title("det = 0 (columns are collinear)")
axes[1].set_xlim(-0.5, 3)
axes[1].set_ylim(-0.5, 3)
axes[1].set_aspect("equal")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

**Geometric interpretation.** The absolute value of the determinant $|\det(A)|$ equals the area of the parallelogram formed by the matrix's column vectors. In the left plot, the two column vectors span a parallelogram with area 3.0. The sign of the determinant indicates whether the transformation preserves or flips orientation. In the right plot, the columns point in the same direction (they are collinear), so the parallelogram collapses to a line segment with zero area — the matrix is singular.

### 1.2 Inverse Matrix

The **inverse** of a matrix $A$ is the matrix $A^{-1}$ such that:

$$A \, A^{-1} = I$$

Think of it as the "undo" operation — like dividing by a number to reverse multiplication. A matrix is invertible only if it is square **and** its determinant is non-zero.

In [ ]:
# Compute inverse and verify A @ A_inv is close to the identity matrix
A = np.array([[2, 1, 0], [1, 3, 1], [0, 2, 1]])
A_inv = np.linalg.inv(A)
print("A inverse:\n", np.round(A_inv, 4))
print("A @ A_inv ≈ I?", np.allclose(A @ A_inv, np.eye(3)))

# Singular matrix — inv raises an error
S = np.array([[1, 2], [2, 4]])
try:
    np.linalg.inv(S)
except np.linalg.LinAlgError as e:
    print("\nCannot invert singular matrix:", e)

### 1.3 Rank

The **rank** of a matrix is the number of linearly independent rows (or columns). It tells you how many dimensions of information the matrix truly carries.

- **Full rank** = no redundant rows or columns.
- For an $m \times n$ matrix, rank $\leq \min(m, n)$.

In [ ]:
# Full rank: all 3 rows are independent
I = np.eye(3)
print("Rank of I:", np.linalg.matrix_rank(I))  # 3

# Rank deficient: row 2 = row 0 + row 1
B = np.array([[1, 2, 3], [4, 5, 6], [5, 7, 9]])
print("Rank of B:", np.linalg.matrix_rank(B))  # 2

# Non-square: rank <= min(m, n)
np.random.seed(0)
C = np.random.randn(4, 3)
print("Rank of 4x3 matrix:", np.linalg.matrix_rank(C))  # 3

### 1.4 How They Connect

For a square $n \times n$ matrix, these three properties are equivalent:

| Condition | Determinant | Invertible? | Rank |
|---|---|---|---|
| Independent columns | $\det(A) \neq 0$ | Yes | Full rank ($= n$) |
| Dependent columns | $\det(A) = 0$ | No | Less than $n$ |

A matrix is invertible **if and only if** its determinant is non-zero **if and only if** it has full rank.

## 2. Solving Linear Systems

### 2.1 From Equations to Matrices

Consider a pair of simultaneous equations:

$$2x + y = 5$$
$$x + 3y = 7$$

This can be written in matrix form $A\mathbf{x} = \mathbf{b}$:

$$\begin{pmatrix} 2 & 1 \\ 1 & 3 \end{pmatrix} \begin{pmatrix} x \\ y \end{pmatrix} = \begin{pmatrix} 5 \\ 7 \end{pmatrix}$$

### 2.2 `np.linalg.solve`

NumPy's `solve` function finds $\mathbf{x}$ directly. It is the recommended way to solve linear systems.

In [ ]:
A = np.array([[2, 1], [1, 3]])
b = np.array([5, 7])

# Solve Ax = b
x = np.linalg.solve(A, b)
print("Solution x =", x)

# Verify: A @ x should equal b
print("A @ x ≈ b?", np.allclose(A @ x, b))

### 2.3 Why `solve` Beats `inv`

You could also solve the system by computing $\mathbf{x} = A^{-1}\mathbf{b}$, but `solve` is preferred because:

- It uses **LU decomposition** internally, which is faster.
- It is **numerically more stable**, especially when the matrix is nearly singular.

The **condition number** `np.linalg.cond(A)` measures how sensitive the solution is to small changes in the input. A large condition number means the system is ill-conditioned.

In [ ]:
# Both give the same answer for a well-conditioned matrix
x_solve = np.linalg.solve(A, b)
x_inv = np.linalg.inv(A) @ b
print("solve:", x_solve)
print("inv:  ", x_inv)

# Ill-conditioned matrix: a tiny change in b causes a huge change in x
A_ill = np.array([[1, 1], [1, 1.0001]])
b_ill = np.array([2, 2.0001])
print("\nCondition number:", f"{np.linalg.cond(A_ill):.0f}")
print("Solution:", np.linalg.solve(A_ill, b_ill))

# Perturb b by a tiny amount (0.0001) and observe the effect on x
b_perturbed = np.array([2, 2.0002])
print("\nPerturbed b by 0.0001:")
print("New solution:", np.linalg.solve(A_ill, b_perturbed))
print("Change in x:", np.linalg.solve(A_ill, b_perturbed) - np.linalg.solve(A_ill, b_ill))
print("A small change in b produced a large change in x — this is ill-conditioning.")

## 3. Eigenvalues and Eigenvectors

### 3.1 The Concept

A matrix represents a linear transformation — it can rotate, stretch, or shear vectors. An **eigenvector** is a special vector whose direction is unchanged by the transformation. The **eigenvalue** is the factor by which it is stretched:

$$A\mathbf{v} = \lambda\mathbf{v}$$

Most vectors change direction when multiplied by a matrix. Eigenvectors are the exception — they only get scaled. Finding them reveals the fundamental axes along which the transformation acts.

### 3.2 Computing with NumPy

In [ ]:
# A simple matrix with integer eigenvalues (2 and 3)
A = np.array([[2, 1], [0, 3]])
eigenvalues, eigenvectors = np.linalg.eig(A)

print("Eigenvalues:", eigenvalues)
print("Eigenvectors (columns):\n", eigenvectors)

# Verify: A @ v should equal lambda * v
for i in range(len(eigenvalues)):
    v = eigenvectors[:, i]   # i-th eigenvector is the i-th COLUMN
    lam = eigenvalues[i]
    print(f"\nEigenvalue {lam:.0f}: A @ v = {A @ v},  lambda * v = {lam * v}")
    print(f"  Match? {np.allclose(A @ v, lam * v)}")

> **Important:** `np.linalg.eig` returns eigenvectors as **columns** — `eigenvectors[:, i]` corresponds to `eigenvalues[i]`. This is a common source of confusion.

### 3.3 Symmetric Matrices

A **symmetric matrix** ($A = A^T$) has two special properties:
- Its eigenvalues are always **real** (no complex numbers).
- Its eigenvectors are **orthogonal** (perpendicular to each other).

Recall that two vectors are orthogonal when their dot product is zero: $\mathbf{u} \cdot \mathbf{v} = 0$. Geometrically, this means they meet at a right angle (90°).

Use `np.linalg.eigh` for symmetric matrices — it is faster and more numerically stable than `eig`.

In [ ]:
A_sym = np.array([[4, 2], [2, 1]])
eigenvalues, eigenvectors = np.linalg.eigh(A_sym)
print("Eigenvalues (real):", eigenvalues)

# Verify orthogonality: dot product of 0 means the vectors are perpendicular
v0, v1 = eigenvectors[:, 0], eigenvectors[:, 1]
print("v0 . v1 =", np.dot(v0, v1))  # approximately 0 -> perpendicular

### 3.4 Eigendecomposition

Any diagonalisable matrix can be decomposed as:

$$A = P \, D \, P^{-1}$$

where $P$ is the matrix of eigenvectors and $D$ is a diagonal matrix of eigenvalues. This allows us to reconstruct the original matrix from its eigenvalues and eigenvectors.

In [ ]:
# Complete code — just run this cell
A = np.array([[4, 2], [1, 3]])
eigenvalues, P = np.linalg.eig(A)

# P: each column is an eigenvector
print("P (eigenvector matrix):\n", np.round(P, 4))

# D: diagonal matrix with eigenvalues on the diagonal
D = np.diag(eigenvalues)
print("\nD (eigenvalue matrix):\n", np.round(D, 4))

# Reconstruct: A = P @ D @ P^(-1)
P_inv = np.linalg.inv(P)
A_reconstructed = P @ D @ P_inv

print("\nOriginal A:\n", A)
print("Reconstructed P @ D @ P^(-1):\n", np.round(A_reconstructed, 6))
print("Match?", np.allclose(A, A_reconstructed))

## 4. Exercises

### Hill Cipher — Matrix Encryption

The Hill cipher encrypts text using matrix multiplication:

1. Map each letter to a number: A=0, B=1, ..., Z=25.
2. Group letters into vectors of length $n$.
3. Multiply each vector by an $n \times n$ key matrix $K$ and take mod 26.
4. To decrypt, use the modular inverse of $K$.

The key matrix must be invertible **and** its determinant must be coprime with 26 (i.e. $\gcd(\det(K), 26) = 1$), otherwise the modular inverse does not exist.

In [ ]:
# Complete code — helper functions for Hill cipher

def text_to_numbers(text):
    """Convert uppercase letters to numbers (A=0, B=1, ..., Z=25)."""
    return np.array([ord(c) - ord('A') for c in text.upper() if c.isalpha()])

def numbers_to_text(numbers):
    """Convert numbers back to uppercase letters."""
    return ''.join(chr(int(n) % 26 + ord('A')) for n in numbers)

def mod_inverse_matrix(K, mod=26):
    """Compute the modular inverse of matrix K under the given modulus."""
    det = int(round(np.linalg.det(K)))
    det_mod = det % mod
    det_inv = pow(det_mod, -1, mod)  # modular inverse of the determinant
    adjugate = np.round(det * np.linalg.inv(K)).astype(int)
    return (det_inv * adjugate) % mod

print("Helper functions loaded.")

#### Encryption

Multiply the plaintext vector by the key matrix $K$ and take mod 26.

In [ ]:
def hill_encrypt(plain_vector, K):
    # ===== YOUR CODE BELOW =====
    cipher_vector = ...
    # ===== YOUR CODE ABOVE =====
    return cipher_vector

In [ ]:
show_hint("hill_encrypt")

In [ ]:
check_hill_encrypt(hill_encrypt)

#### Key Validation

A key matrix $K$ is valid for the Hill cipher only if its determinant (mod 26) is non-zero **and** coprime with 26.

In [ ]:
def is_valid_key(K):
    det = int(round(np.linalg.det(K))) % 26
    # ===== YOUR CODE BELOW =====
    valid = ...
    # ===== YOUR CODE ABOVE =====
    return valid

In [ ]:
show_hint("valid_key")

In [ ]:
check_valid_key(is_valid_key)

#### Decryption

Decrypt by applying the modular inverse of the key matrix, exactly as you did for encryption.

In [ ]:
def hill_decrypt(cipher_vector, K):
    K_inv = mod_inverse_matrix(K, 26)  # provided helper
    # ===== YOUR CODE BELOW =====
    plain_vector = ...
    # ===== YOUR CODE ABOVE =====
    return plain_vector

In [ ]:
show_hint("hill_decrypt")

In [ ]:
check_hill_decrypt(hill_decrypt)

In [ ]:
# Complete code — encrypt and decrypt a message
K = np.array([[3, 3], [2, 5]])
message = "HELLO"

# Our 2x2 key processes letters in pairs. "HELLO" has 5 letters (odd),
# so we pad with X (=23) to make it even: "HELLOX". This is why the
# decrypted output ends with an extra X.
nums = text_to_numbers(message)
if len(nums) % 2 != 0:
    nums = np.append(nums, 23)  # pad with X to make length even

print(f"Plaintext:  {message}")
print(f"Numbers:    {nums}  (padded to even length with X=23)")

# Encrypt in pairs
cipher_nums = np.concatenate([hill_encrypt(nums[i:i+2], K) for i in range(0, len(nums), 2)])
print(f"Encrypted:  {numbers_to_text(cipher_nums)}  ({cipher_nums.astype(int)})")

# Decrypt back — the trailing X is padding, not part of the original message
plain_nums = np.concatenate([hill_decrypt(cipher_nums[i:i+2], K) for i in range(0, len(cipher_nums), 2)])
print(f"Decrypted:  {numbers_to_text(plain_nums)}  (trailing X is padding)")

### PageRank — Ranking with Eigenvectors

Google's original algorithm ranks web pages by importance. The core idea: a page is important if many important pages link to it. This recursive definition is exactly what eigenvectors solve.

We build a **transition matrix** from the link structure, add a damping factor $d = 0.85$ (modelling the chance a user follows a link rather than jumping to a random page), and find the dominant eigenvector — that is the PageRank vector.

In [ ]:
# Complete code — define the web graph and visualise it
pages = ['A', 'B', 'C', 'D', 'E', 'F']
edges = [('A','B'), ('A','C'), ('B','C'), ('C','A'),
         ('D','C'), ('E','C'), ('E','D'), ('F','D'), ('F','E')]

G_graph = nx.DiGraph()
G_graph.add_nodes_from(pages)
G_graph.add_edges_from(edges)

# Fixed positions: A-B-C form a triangle on the left, D-E-F on the right
# Arrows flow visibly between the two clusters
pos = {
    'A': (0, 1),   'B': (1, 2),   'C': (1, 0),
    'D': (3, 0),   'E': (3, 2),   'F': (4, 1),
}

plt.figure(figsize=(7, 4))
nx.draw(G_graph, pos, with_labels=True, node_color="lightblue",
        node_size=700, font_size=14, font_weight="bold",
        arrows=True, arrowsize=18, edge_color="grey",
        connectionstyle="arc3,rad=0.1")
plt.title("Web graph: 6 pages, 9 directed links")
plt.show()

In [ ]:
# Complete code — build the Google matrix
n = len(pages)

# Adjacency matrix: adj[j, i] = 1 means page i links to page j
adj = np.zeros((n, n))
page_idx = {p: i for i, p in enumerate(pages)}
for src, dst in edges:
    adj[page_idx[dst], page_idx[src]] = 1

# Column-normalise: each column sums to 1
col_sums = adj.sum(axis=0)
col_sums[col_sums == 0] = 1  # avoid division by zero
M = adj / col_sums

# Apply damping factor
d = 0.85
G = d * M + (1 - d) / n * np.ones((n, n))

print("Google matrix G:\n", np.round(G, 3))

#### Compute PageRank via Eigenvectors

The PageRank vector is the eigenvector corresponding to the largest eigenvalue of the Google matrix $G$. Normalise it so all entries sum to 1.

In [ ]:
def compute_pagerank(G):
    eigenvalues, eigenvectors = np.linalg.eig(G)
    # ===== YOUR CODE BELOW =====
    # Find the eigenvector for the largest eigenvalue and normalise it
    idx = ...
    rank_vector = ...
    rank_vector = ...
    # ===== YOUR CODE ABOVE =====
    return np.real(rank_vector)  # discard tiny imaginary parts

In [ ]:
show_hint("pagerank_eigen")

In [ ]:
check_pagerank(compute_pagerank)

#### Compute PageRank via Power Iteration

An alternative approach: start with a uniform distribution and repeatedly multiply by $G$. The vector converges to the PageRank.

In [ ]:
def pagerank_power_iteration(G, num_iterations=100):
    n = G.shape[0]
    r = np.ones(n) / n  # start with uniform distribution
    for _ in range(num_iterations):
        # ===== YOUR CODE BELOW =====
        r = ...
        # ===== YOUR CODE ABOVE =====
    return r

In [ ]:
show_hint("pagerank_power")

In [ ]:
check_power_iteration(pagerank_power_iteration)

In [ ]:
# Complete code — visualise the rankings
ranks = compute_pagerank(G)

# Draw graph with node sizes proportional to PageRank
plt.figure(figsize=(7, 4))
node_sizes = 3000 * ranks / ranks.max()
nx.draw(G_graph, pos, with_labels=True, node_size=node_sizes,
        node_color=ranks, cmap=plt.cm.Reds,
        font_size=14, font_weight="bold",
        arrows=True, arrowsize=18, edge_color="grey",
        connectionstyle="arc3,rad=0.1")
plt.title("PageRank — node size and colour = importance")
plt.show()

# Print ranking table sorted by importance
order = np.argsort(-ranks)
print("Page  |  PageRank")
print("------+---------")
for i in order:
    print(f"  {pages[i]}   |  {ranks[i]:.4f}")

In [ ]:
# Complete code — what happens if we add an edge?
# The highest-ranked page (C) now links to the lowest-ranked page (F).
# This should boost F's rank because it receives a link from an important page.
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Original graph
top_page = pages[np.argmax(ranks)]
bottom_page = pages[np.argmin(ranks)]

axes[0].set_title("Before: original graph")
nx.draw(G_graph, pos, ax=axes[0], with_labels=True,
        node_size=3000 * ranks / ranks.max(),
        node_color=ranks, cmap=plt.cm.Reds,
        font_size=14, font_weight="bold",
        arrows=True, arrowsize=18, edge_color="grey",
        connectionstyle="arc3,rad=0.1")

# Add edge from highest-ranked to lowest-ranked page
G_graph2 = G_graph.copy()
G_graph2.add_edge(top_page, bottom_page)

# Rebuild Google matrix with the new edge
adj2 = adj.copy()
adj2[page_idx[bottom_page], page_idx[top_page]] = 1
col_sums2 = adj2.sum(axis=0)
col_sums2[col_sums2 == 0] = 1
M2 = adj2 / col_sums2
G2 = d * M2 + (1 - d) / n * np.ones((n, n))
ranks2 = compute_pagerank(G2)

axes[1].set_title(f"After: added {top_page} -> {bottom_page}")
nx.draw(G_graph2, pos, ax=axes[1], with_labels=True,
        node_size=3000 * ranks2 / ranks2.max(),
        node_color=ranks2, cmap=plt.cm.Reds,
        font_size=14, font_weight="bold",
        arrows=True, arrowsize=18, edge_color="grey",
        connectionstyle="arc3,rad=0.1")

plt.show()

# Show how each page's rank changed
print("Page  |  Before  |  After   |  Change")
print("------+----------+----------+--------")
for i in range(n):
    diff = ranks2[i] - ranks[i]
    print(f"  {pages[i]}   |  {ranks[i]:.4f}  |  {ranks2[i]:.4f}  |  {diff:+.4f}")

In [ ]:
# Complete code — convergence of power iteration
r = np.ones(n) / n
history = [r.copy()]
for _ in range(20):
    r = G @ r
    history.append(r.copy())
history = np.array(history)

plt.figure(figsize=(8, 4))
for i in range(n):
    plt.plot(history[:, i], label=pages[i])
plt.xlabel("Iteration")
plt.ylabel("Rank score")
plt.title("Convergence of power iteration")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Summary

In practice, Google uses power iteration rather than eigenvalue decomposition — the internet has billions of pages, and full eigendecomposition of such a large matrix is infeasible. Power iteration only requires repeated matrix-vector multiplication, which is efficient for large sparse matrices.

This echoes a recurring theme in numerical computing: choosing the right algorithm matters as much as knowing the mathematics.